# patch stitching error with overlap
Projecting down the results from 3 to 1 dimension using PCA. Then correlating the result with the individual input variables.

1. load model
2. load dataset patches
3. predict results using NOFE and other methods
4. stitch results together + plot
5. quantitative evaluation based on neighbors across patches

---
### Utilities

In [ ]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/modis_himalaya/scripts/")
from plot_utilities import *
from utilities_modis_himalaya import MODIS_Himalaya_Graph_Generator as Data_Generator
data_path = project_root / 'modis_himalaya' / 'data' / 'himalaya'

device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

from scipy.spatial import cKDTree

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'modis_himalaya' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'modis_himalaya' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    with open(config_path, 'r') as f:
        config = json.load(f)
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model, config

def mean_dist(x):
    N = x.shape[0]
    dists = torch.cdist(x, x)
    mean_dist = dists.triu(diagonal=1).sum() / (N * (N - 1) / 2)
    return mean_dist

----
### load model + dataset 

In [ ]:
model, config = load_model(id='testtraining2_2026-05-05_t08-38-27', model_type='regular')

In [ ]:
bbox = config["bbox"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
feature_mode = config["feature_mode"]
min_file_size_mb = config.get("min_file_size_mb", 150)
print(f'bbox: {bbox}')
print(f'node_features: {config["node_features"]}')

In [ ]:
row_min, col_min, row_max, col_max = bbox
row_range = (row_max - row_min)/2
col_range = (col_max - col_min)/2
margin=5

# 4 patches with overlap (similar to ERA5 version)
bbox1 = (row_min, col_min+col_range-margin, row_max-col_range+margin, row_max)
bbox2 = (row_min, col_min, row_max, col_min+col_range+margin)
bbox3 = (row_min+row_range-margin, col_min, row_max, col_min+col_range+margin)
bbox4 = (row_min+row_range-margin, col_min+col_range-margin, row_max, col_max)

bboxes = [bbox1, bbox2, bbox3, bbox4]
print(f'Patches with overlap defined: {len(bboxes)}')
for i, b in enumerate(bboxes):
    print(f'  bbox{i+1}: {b}')

In [ ]:
# Get available timestamps, sort by file size, select N largest
Generator = Data_Generator(data_path)
all_timestamps = Generator.get_available_timestamps(
    min_date='2017-01-01',
    max_date='2018-12-31',
    min_file_size_mb=min_file_size_mb
)
timestamp_info = []
for ts in all_timestamps:
    bright_path = os.path.join(Generator.directory, f'brighttemp_1KM_grid_modis_{ts}.tif')
    size = os.path.getsize(bright_path)
    timestamp_info.append((ts, size))
timestamp_info.sort(key=lambda x: x[1], reverse=True)
N_SAMPLES = 30
dates = [ts for ts, _ in timestamp_info[:N_SAMPLES]]
print(f'Selected {len(dates)} timestamps from {len(all_timestamps)} available')
print(f'Date range: {dates[0][:8]} to {dates[-1][:8]}')

N=10000
print('generate data')
graphs=[]
locations_list=[]
X_list=[]

# For overlap case, generate data for each timestamp using the first bbox only
for date in dates[:1]:  # Start with first timestamp for testing
    for bbox in bboxes:
        Generator.load_data(timestamp=date, bbox=bbox)
        print(f'Total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
        graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
        sample = graph.to(device)
        graphs.append(sample)
        locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
        X_list.append(sample.normed_values.detach().cpu().numpy())

# Predict (NOFE, PCA, t-SNE, UMAP)

In [ ]:
pred_nofe_list = []
pred_pca_list = []
pred_tsne_list = []
pred_umap_list = []
locations_list=[]
for sample in graphs:
    locations_list.append(sample.meta.gps)
    print('NOFE ...', end='\r')
    pred_nofe = model(sample).detach().cpu().numpy()
    pred_nofe_list.append(pred_nofe)
    X = sample.normed_values.detach().cpu().numpy()
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pred_pca_list.append(pred_pca)
    print('t-SNE ...', end='\r')
    pred_tsne = TSNE(n_components=3, perplexity=30, learning_rate="auto", random_state=42).fit_transform(X)
    pred_tsne_list.append(pred_tsne)
    print('UMAP ...', end='\r')
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    os.makedirs(dir, exist_ok=True)
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    pred_umap_list.append(pred_umap)
    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')


# Plot

In [ ]:
locations = pd.DataFrame(np.vstack(locations_list), columns=['lat', 'lon'])
pred_nofe_combined = pd.DataFrame(np.vstack(pred_nofe_list), columns=['x1', 'x2', 'x3'])
pred_nofe_combined = pd.concat([locations, pred_nofe_combined], axis=1)
pred_pca_combined = pd.DataFrame(np.vstack(pred_pca_list), columns=['x1', 'x2', 'x3'])
pred_pca_combined = pd.concat([locations, pred_pca_combined], axis=1)
pred_tsne_combined = pd.DataFrame(np.vstack(pred_tsne_list), columns=['x1', 'x2', 'x3'])
pred_tsne_combined = pd.concat([locations, pred_tsne_combined], axis=1)
pred_umap_combined = pd.DataFrame(np.vstack(pred_umap_list), columns=['x1', 'x2', 'x3'])
pred_umap_combined = pd.concat([locations, pred_umap_combined], axis=1)
figsize=(5,5)
percentile=.5
fig_nofe = rgb_scale(pred_nofe_combined, figsize=figsize, percentile=percentile)
fig_pca = rgb_scale(pred_pca_combined, figsize=figsize, percentile=percentile)
fig_tsne = rgb_scale(pred_tsne_combined, figsize=figsize, percentile=percentile)
fig_umap = rgb_scale(pred_umap_combined, figsize=figsize, percentile=percentile)


# Quantitative Evaluation (with Overlap)

In [ ]:
edge_margin = 5  # Larger margin for overlap case
idx12 = np.where(graphs[0].meta.gps[:,1] < (bbox1[0]+edge_margin))[0]
idx21 = np.where(graphs[1].meta.gps[:,1] > (bbox2[2]-edge_margin))[0]
idx13 = np.where(graphs[0].meta.gps[:,0] > (bbox1[3]-edge_margin))[0]
idx31 = np.where(graphs[2].meta.gps[:,0] < (bbox3[1]+edge_margin))[0]
idx24 = np.where(graphs[1].meta.gps[:,0] > (bbox2[3]-edge_margin))[0]
idx42 = np.where(graphs[3].meta.gps[:,0] < (bbox4[1]+edge_margin))[0]
idx34 = np.where(graphs[2].meta.gps[:,1] < (bbox3[0]+edge_margin))[0]
idx43 = np.where(graphs[3].meta.gps[:,1] > (bbox4[2]-edge_margin))[0]
edge_idx_pairs = [(idx12, idx21), (idx13,idx31), (idx24,idx42), (idx34,idx43)]
edge_subset_pairs = [(0, 1), (0,2), (1,3), (2,3)]
stitching_errors_nofe = []
stitching_errors_pca = []
stitching_errors_tsne = []
stitching_errors_umap = []
methods = ['nofe', 'pca', 'tsne', 'umap']
for method_id in range(4):
    if methods[method_id] == 'nofe':
        output_data = pred_nofe_list
    elif methods[method_id] == 'pca':
        output_data = pred_pca_list
    elif methods[method_id] == 'tsne':
        output_data = pred_tsne_list
    elif methods[method_id] == 'umap':
        output_data = pred_umap_list
    normed_distance_tensors = []
    for i in range(4):
        a = edge_subset_pairs[i][0]
        b = edge_subset_pairs[i][1]
        a_idx = edge_idx_pairs[i][0]
        b_idx = edge_idx_pairs[i][1]
        coords_a = graphs[a].normed_coords[a_idx,:].detach().cpu().numpy()
        coords_b = graphs[b].normed_coords[b_idx,:].detach().cpu().numpy()
        tree_a = cKDTree(coords_b)
        distances_a, indices_a = tree_a.query(coords_a)
        A = torch.from_numpy(output_data[a][a_idx])
        B = torch.from_numpy(output_data[b][b_idx][indices_a])
        normalization = mean_dist(A)
        normed_dist = torch.norm(A - B, dim=1)/normalization
        normed_distance_tensors.append(normed_dist)
        a = edge_subset_pairs[i][0]
        b = edge_subset_pairs[i][1]
        a_idx = edge_idx_pairs[i][0]
        b_idx = edge_idx_pairs[i][1]
        coords_a = graphs[a].normed_coords[a_idx,:].detach().cpu().numpy()
        coords_b = graphs[b].normed_coords[b_idx,:].detach().cpu().numpy()
        tree_b = cKDTree(coords_a)
        distances_b, indices_b = tree_b.query(coords_b)
        A = torch.from_numpy(output_data[a][a_idx][indices_b])
        B = torch.from_numpy(output_data[b][b_idx])
        normed_dist = torch.norm(A - B, dim=1)/normalization
        normed_distance_tensors.append(normed_dist)
    if methods[method_id] == 'nofe':
        stitching_errors_nofe = torch.cat(normed_distance_tensors).mean().item()
    elif methods[method_id] == 'pca':
        stitching_errors_pca = torch.cat(normed_distance_tensors).mean().item()
    elif methods[method_id] == 'tsne':
        stitching_errors_tsne = torch.cat(normed_distance_tensors).mean().item()
    elif methods[method_id] == 'umap':
        stitching_errors_umap = torch.cat(normed_distance_tensors).mean().item()
print(f'Stitching errors with overlap (margin={edge_margin}):')
print(f'nofe: {stitching_errors_nofe}, pca: {stitching_errors_pca}, tsne: {stitching_errors_tsne}, umap: {stitching_errors_umap}')
